# Bibliotecas

In [2]:
import pandas as pd
import numpy as np
import glob
import os

## CARREGAMENTO, LIMPEZA E TRATAMENTO DOS ARQUIVOS INDIVIDUALMENTE

In [ ]:
# Realizar esse processo para cada dataset da pasta uploads
arquivo_csv = ('uploads/processos_6.csv')
df = pd.read_csv(arquivo_csv, sep='#', encoding='utf-8-sig')
print(df.shape)

## VALIDAÇÃO RÁPIDA

In [ ]:
pendentes = df['data_baixa'].isna().sum()
print(f"Pendentes: {pendentes}")

In [ ]:
baixados = df['data_baixa'].notna().sum()
print(f"Baixados: {baixados}")

In [ ]:
tx_cong_global = (100 * pendentes/(pendentes+baixados)).round(2)
print(f"Taxa de Congestionamento Global: {tx_cong_global} %")

In [ ]:
# Verificando se na coluna data_baixa e data_distribuicao tem o ANO e quantos são:
# Converte a coluna para o tipo datetime
ANO = 2026
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce') # errors='coerce' transforma erros em NaT (nulo)
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce') 

print("="*60)
print("--- VALIDAÇÃO DE DATA_BAIXA ---")
# Faz a verificação
temp = (df['data_baixa'].dt.year == ANO).any()
print(temp)

# Se quiser saber QUANTOS registros existem com o ano 2019:
qtd = (df['data_baixa'].dt.year == ANO).sum()
print(f"Total de registros de data_baixa em {ANO}: {qtd}")

print("="*60)
print("--- VALIDAÇÃO DE DATA_DISTRIBUICAO ---")
# Faz a verificação
temp2 = (df['data_distribuicao'].dt.year == ANO).any()
print(temp2)

# Se quiser saber QUANTOS registros existem com o ano 2019:
qtd2 = (df['data_distribuicao'].dt.year == ANO).sum()
print(f"Total de registros de data_distribuicao em {ANO}: {qtd2}")

## LIMPEZA E TRATAMENTO DAS DATAS DO ARQUIVO_CSV

In [ ]:
# Limpeza e tratamento das datas do arquivo_csv
# 1. Converter as colunas de data para datetime
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce')
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce')

# 2. Verificar quantas datas inválidas temos
print(f"\nDatas inválidas na conversão:")
print(f"- data_distribuicao: {df['data_distribuicao'].isna().sum()} valores NaT")
print(f"- data_baixa: {df['data_baixa'].isna().sum()} valores NaT")

# 3. Extrair o ano de cada data
df['ano_distribuicao'] = df['data_distribuicao'].dt.year
df['ano_baixa'] = df['data_baixa'].dt.year

# 4. Fazer a limpeza: manter apenas registros em que:
#    - ano_distribuicao <= ANO (ou é NaN)
#    - ano_baixa <= ANO (ou é NaN)
ANO = 2025
df_limpo = df[
    (
        (df['ano_distribuicao'] <= ANO)
    ) & 
    (
        (df['ano_baixa'] <= ANO) | 
        (df['ano_baixa'].isna())
    )
].copy()

print(f"\nShape após limpeza: {df_limpo.shape}")
print(f"Registros removidos: {len(df) - len(df_limpo)}")

# 5. Verificação detalhada
print(f"\nVerificação dos anos nas datas:")
print(f"Anos únicos em data_distribuicao: {sorted(df_limpo['ano_distribuicao'].dropna().unique().astype(int))}")
print(f"Anos únicos em data_baixa: {sorted(df_limpo['ano_baixa'].dropna().unique().astype(int))}")

# 6. Mostrar exemplos de registros removidos (se houver)
if len(df) > len(df_limpo):
    df_removidos = df[~df.index.isin(df_limpo.index)]
    print(f"\nExemplo de registros removidos:")
    print(df_removidos[['data_distribuicao', 'data_baixa', 'ano_distribuicao', 'ano_baixa']].head())
    
    # Contar motivo da remoção
    print(f"\nMotivo da remoção:")
    # Distribuição > ANO
    ANO = 2025
    dist_maior_ANO = df[df['ano_distribuicao'] > ANO]
    dist_nula = df[df['ano_distribuicao'].isna()]
    baixa_maior_ANO = df[df['ano_baixa'] > ANO]
    
    print(f"- data_distribuicao > {ANO}: {len(dist_maior_ANO)} registros")
    print(f"- data_distribuicao NULA: {len(dist_nula)} registros")
    print(f"- data_baixa > {ANO}: {len(baixa_maior_ANO)} registros")
    
    # Intersecção (ambos > ANO)
    ambos_maior_ANO = df[(df['ano_distribuicao'] > ANO) & (df['ano_baixa'] > ANO)]
    print(f"- ambos > {ANO}: {len(ambos_maior_ANO)} registros")

# 7. Opcional: Remover as colunas auxiliares de ano se não precisar mais
# df_limpo = df_limpo.drop(columns=['ano_distribuicao', 'ano_baixa'])

# 8. Mostrar estatísticas do dataframe limpo
print(f"\n{'='*50}")
print("RESUMO DO DATAFRAME LIMPO:")
print(f"{'='*50}")
print(f"Total de registros: {len(df_limpo)}")
print(f"Total de colunas: {len(df_limpo.columns)}")
print(f"\nTipos de dados:")
print(df_limpo.dtypes)

# 9. Verificar valores nulos nas datas
print(f"\nValores nulos nas datas (após limpeza):")
print(f"- data_distribuicao: {df_limpo['data_distribuicao'].isna().sum()} nulos ({df_limpo['data_distribuicao'].isna().sum()/len(df_limpo)*100:.1f}%)")
print(f"- data_baixa: {df_limpo['data_baixa'].isna().sum()} nulos ({df_limpo['data_baixa'].isna().sum()/len(df_limpo)*100:.1f}%)")

In [ ]:
# Validação da Limpeza do dataframe
ANO = 2026
df_limpo.info()
df_limpo_distribuicao = df_limpo[(df_limpo['ano_distribuicao'] == ANO)].head(10)
df_limpo_baixa = df_limpo[(df_limpo['ano_baixa'] == ANO)].head(10)

print('###################### data_distribuicao com ano = ANO ######################')
print(df_limpo_distribuicao)
print('######################### data_baixa com ano = ANO ##########################')
print(df_limpo_baixa)

## GERAR OS ARQUIVOS `clean_process_*.csv` NA PASTA `dataclean`

In [ ]:
output_filename_csv_mes = 'dataclean/clean_process_6.csv'
os.makedirs(os.path.dirname(output_filename_csv_mes), exist_ok=True)
df_limpo.to_csv(output_filename_csv_mes, index=False, sep='#', encoding='utf-8-sig')

# Carregamento de TODOS os Arquivos CSV Limpos

In [3]:
# Concatena os arquivos LIMPOS da pasta dataclean
# Listar os arquivos CSV na pasta 'dataclean'
arquivo_csv = glob.glob('dataclean/clean_process_*.csv')
# Carregar os arquivos CSV e concatenar em um único DataFrame
dfs = []
for arquivo in arquivo_csv:  # lista/iterável com os caminhos tipo 'clean_process_1.csv', 'clean_process_2.csv', ...
    df_ano = pd.read_csv(arquivo, sep='#', encoding='utf-8-sig')
    dfs.append(df_ano)

df = pd.concat(dfs, ignore_index=True)

print(df.shape)

C:\Users\jcpsrodrigues\AppData\Local\Temp\ipykernel_16508\3153771756.py:7: DtypeWarning: Columns (5,19,20,24,25,41,43,54,56,60,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ano = pd.read_csv(arquivo, sep='#', encoding='utf-8-sig')
C:\Users\jcpsrodrigues\AppData\Local\Temp\ipykernel_16508\3153771756.py:7: DtypeWarning: Columns (5,19,20,22,23,25,41,43,49,54,56,60,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ano = pd.read_csv(arquivo, sep='#', encoding='utf-8-sig')
C:\Users\jcpsrodrigues\AppData\Local\Temp\ipykernel_16508\3153771756.py:7: DtypeWarning: Columns (5,19,20,24,25,41,43,49,54,56,60,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ano = pd.read_csv(arquivo, sep='#', encoding='utf-8-sig')
C:\Users\jcpsrodrigues\AppData\Local\Temp\ipykernel_16508\3153771756.py:7: DtypeWarning: Columns (5,20,41,43,54,60,61) have mixed types. Specify dtype option on import or set low_memory=F

(11649349, 66)


## Ordenar e Remover Duplicatas

In [4]:
# Ordenar para garantir que os registros mais recentes (com baixa preenchida) fiquem por último

# Isso garante que se um processo era nulo no df1 e foi baixado no df2, a versão baixada prevaleça.
df = df.sort_values(by=['processo_id', 'data_baixa'], na_position='first')

# Remover duplicatas, mantendo a última ocorrência (a mais atualizada)
df = df.drop_duplicates(subset=['processo_id'], keep='last').copy()

# Validações Rápidas

In [5]:
pendentes = df['data_baixa'].isna().sum()
print(f"Pendentes: {pendentes}")

Pendentes: 2837154


In [6]:
baixados = df['data_baixa'].notna().sum()
print(f"Baixados: {baixados}")

Baixados: 3549959


In [7]:
tx_cong_global = (100 * pendentes/(pendentes+baixados)).round(2)
print(f"Taxa de Congestionamento Global: {tx_cong_global} %")

Taxa de Congestionamento Global: 44.42 %


# Limpeza e Tratamento

In [8]:
# Verificar o nome correto das colunas (pode haver diferenças de acentuação ou espaços)
colunas = df.columns.tolist()

# Encontrar as colunas de data corretamente
coluna_serventia = [col for col in colunas if 'serventia' in col.lower()][0]
coluna_distribuicao = [col for col in colunas if 'data_distribuicao' in col.lower()][0]
coluna_baixa = [col for col in colunas if 'data_baixa' in col.lower()][0]
coluna_area_acao = [col for col in colunas if 'nome_area_acao' in col.lower()][0]
coluna_processo_id = [col for col in colunas if 'processo_id' in col.lower()][0]
coluna_comarca = [col for col in colunas if 'comarca' in col.lower()][0]

# Renomear colunas para garantir consistência
df = df.rename(columns={
coluna_distribuicao: 'data_distribuicao',
coluna_baixa: 'data_baixa',
coluna_area_acao: 'nome_area_acao',
coluna_processo_id: 'processo_id',
coluna_comarca: 'comarca',
coluna_serventia: 'serventia'
})

# ==========================================================================================================================================
# ANÁLISE MENSAL
# ==========================================================================================================================================

In [9]:
# Análise Mensal
# Gerando o Dataframe com a Taxa de Congestionamento Mensal:
# 0) Garantir datetime
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce')
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce')

keys = ['comarca', 'serventia'] # Altere aqui para fazer os agrupamentos

# 1) Cópia 
df_f = df.copy()

# 2.1) Referência mensal
df_f['mes_dist'] = df_f['data_distribuicao'].dt.to_period('M')
df_f['mes_baixa'] = df_f['data_baixa'].dt.to_period('M')

# 3.1) Fluxos mensais
# Distribuídos por mês (fluxo)
dist_mes_base = (
    df_f.groupby(keys + ['mes_dist'])
        .size()
        .rename('Distribuidos_fluxo_mensal')
        .reset_index()
        .rename(columns={'mes_dist': 'mes_ref'})
)

# Baixados por mês (fluxo)
baix_mes_base = (
    df_f.dropna(subset=['data_baixa'])
        .groupby(keys + ['mes_baixa'])
        .size()
        .rename('Baixados_fluxo_mensal')
        .reset_index()
        .rename(columns={'mes_baixa': 'mes_ref'})
)

# Calcular valores acumulados
# Distribuídos acumulados
dist_acum = dist_mes_base.copy()
dist_acum = dist_acum.sort_values(['comarca', 'serventia', 'mes_ref'])
dist_acum['Distribuidos_acum_mes'] = dist_acum.groupby(keys)['Distribuidos_fluxo_mensal'].cumsum()

# Baixados acumulados
baix_acum = baix_mes_base.copy()
baix_acum = baix_acum.sort_values(['comarca', 'serventia', 'mes_ref'])
baix_acum['Baixados_acum_mes'] = baix_acum.groupby(keys)['Baixados_fluxo_mensal'].cumsum()

# Pendentes acumulados = Distribuídos acumulados - Baixados acumulados
pend_acum = dist_acum[keys + ['mes_ref', 'Distribuidos_acum_mes']].copy()
pend_acum = pend_acum.merge(
    baix_acum[keys + ['mes_ref', 'Baixados_acum_mes']], 
    on=keys + ['mes_ref'], 
    how='left'
)
pend_acum['Baixados_acum_mes'] = pend_acum['Baixados_acum_mes'].fillna(0).astype(int)
pend_acum['Pendentes_acum_mes'] = pend_acum['Distribuidos_acum_mes'] - pend_acum['Baixados_acum_mes']

# 6.1) Dataframe mensal
df_global_mensal = dist_mes_base[['comarca', 'serventia', 'mes_ref', 'Distribuidos_fluxo_mensal']].copy()

# Adicionar Baixados_fluxo_mensal
df_global_mensal = df_global_mensal.merge(
    baix_mes_base[['comarca', 'serventia', 'mes_ref', 'Baixados_fluxo_mensal']],
    on=['comarca', 'serventia', 'mes_ref'],
    how='left'
)

# Adicionar Pendentes_acum_mes
df_global_mensal = df_global_mensal.merge(
    pend_acum[['comarca', 'serventia', 'mes_ref', 'Pendentes_acum_mes']],
    on=['comarca', 'serventia', 'mes_ref'],
    how='left'
)

# Adicionar Distribuidos_acum_mes (apenas para cálculo, depois removemos)
df_global_mensal = df_global_mensal.merge(
    dist_acum[['comarca', 'serventia', 'mes_ref', 'Distribuidos_acum_mes']],
    on=['comarca', 'serventia', 'mes_ref'],
    how='left'
)

# Preencher NaN com 0 e Converter para int
df_global_mensal['Baixados_fluxo_mensal'] = df_global_mensal['Baixados_fluxo_mensal'].fillna(0).astype(int)
df_global_mensal['Pendentes_acum_mes'] = df_global_mensal['Pendentes_acum_mes'].fillna(0).astype(int)
df_global_mensal['Distribuidos_fluxo_mensal'] = df_global_mensal['Distribuidos_fluxo_mensal'].astype(int)
df_global_mensal['Distribuidos_acum_mes'] = df_global_mensal['Distribuidos_acum_mes'].astype(int)

# Renomear colunas
df_global_mensal = df_global_mensal.rename(columns={
    'Distribuidos_fluxo_mensal': 'Distribuidos_mes',
    'Baixados_fluxo_mensal': 'Baixados_mes',
    'Pendentes_acum_mes': 'Pendentes_mes'
})

# Calcular taxa mensal CORRETA: Pendentes_mes / Distribuidos_acum_mes
df_global_mensal['Taxa_Cong_mensal (%)'] = np.where(
    df_global_mensal['Distribuidos_acum_mes'] > 0,
    (df_global_mensal['Pendentes_mes'] / df_global_mensal['Distribuidos_acum_mes']) * 100,
    0
).round(2)

# Remover a coluna auxiliar Distribuidos_acum_mes
df_global_mensal = df_global_mensal.drop('Distribuidos_acum_mes', axis=1)

# Ordenar por comarca, serventia e mês
df_global_mensal = df_global_mensal.sort_values(['comarca', 'serventia', 'mes_ref'])

In [10]:
# Filtrando todos os MESES de cada ano para exração MENSAL
ano_alvo = 2020
df_mes = df_global_mensal[df_global_mensal['mes_ref'].dt.year == ano_alvo]
df_mes.head(12)

,comarca,serventia,mes_ref,Distribuidos_mes,Baixados_mes,Pendentes_mes,Taxa_Cong_mensal (%)
205,ABADIÂNIA,Vara Judicial,2020-01,98,150,4686,96.90
206,ABADIÂNIA,Vara Judicial,2020-02,89,41,4734,96.12
207,ABADIÂNIA,Vara Judicial,2020-03,147,81,4800,94.64
208,ABADIÂNIA,Vara Judicial,2020-04,37,30,4807,94.09
209,ABADIÂNIA,Vara Judicial,2020-05,64,42,4829,93.35
210,ABADIÂNIA,Vara Judicial,2020-06,103,43,4889,92.66
211,ABADIÂNIA,Vara Judicial,2020-07,119,35,4973,92.18
212,ABADIÂNIA,Vara Judicial,2020-08,93,46,5020,91.47
213,ABADIÂNIA,Vara Judicial,2020-09,136,44,5112,90.90
214,ABADIÂNIA,Vara Judicial,2020-10,86,24,5174,90.61


In [11]:
filtro = (
    (df_global_mensal['comarca'] == 'ABADIÂNIA') & 
    (df_global_mensal['serventia'] == 'Vara Judicial') &
    (df_global_mensal['mes_ref'].astype(str).str.contains('2021'))
)
df_global_mensal[filtro].head(12)

,comarca,serventia,mes_ref,Distribuidos_mes,Baixados_mes,Pendentes_mes,Taxa_Cong_mensal (%)
217,ABADIÂNIA,Vara Judicial,2021-01,96,41,5518,89.32
218,ABADIÂNIA,Vara Judicial,2021-02,115,93,5540,88.03
219,ABADIÂNIA,Vara Judicial,2021-03,132,58,5614,87.38
220,ABADIÂNIA,Vara Judicial,2021-04,160,31,5743,87.21
221,ABADIÂNIA,Vara Judicial,2021-05,180,44,5879,86.90
222,ABADIÂNIA,Vara Judicial,2021-06,197,28,6048,86.87
223,ABADIÂNIA,Vara Judicial,2021-07,179,73,6154,86.18
224,ABADIÂNIA,Vara Judicial,2021-08,292,96,6350,85.43
225,ABADIÂNIA,Vara Judicial,2021-09,209,69,6490,84.93
226,ABADIÂNIA,Vara Judicial,2021-10,230,91,6629,84.21


In [ ]:
# Grava o arquivo MENSAL concatenado, no formato JSON, na pasta results_concat
#df_global_mensal.to_json("results_concat/tx_cong_mensal.json", orient="records", force_ascii=False)